# A/B Test: New Landing Page vs Control (GP3)

**Фокус:** подготовка данных A/B-теста, проверка корректности дизайна, расчёт конверсии и статистический вывод.  

Датасет загружается из Google Drive.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest

plt.rcParams["figure.figsize"] = (10, 5)

## 1) Загрузка данных

> Источник — ссылка на CSV (из учебного задания). Если ссылка недоступна — можно скачать файл локально и заменить `url` на путь к файлу.


In [ ]:
url = "https://drive.google.com/uc?id=1-8ZoKghCxGMBAIWSZV2JT0dymPdNPN4i"
df = pd.read_csv(url)
df.head()

## 2) Проверка качества данных

Проверяем:
- пропуски
- дубликаты по пользователю (`user_id`)
- корректность бинарных полей
- mismatch: когда `group` не соответствует `landing_page`


In [ ]:
df.isna().sum()

In [ ]:
df['user_id'].duplicated().sum()

In [ ]:
df['converted'].value_counts(dropna=False), df['group'].value_counts(dropna=False), df['landing_page'].value_counts(dropna=False)

In [ ]:
pd.crosstab(df['group'], df['landing_page'])

## 3) Очистка данных под A/B-тест

Правила:
1) Удаляем mismatch (control должен видеть old_page, treatment — new_page)  
2) Удаляем пользователей, попавших в обе группы (оставляем “чистые” наблюдения)  
3) Удаляем дубликаты `user_id`


In [ ]:
# 1) убираем mismatch
mask_correct = (
    ((df['group'] == 'control') & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
)
df_clean = df.loc[mask_correct].copy()

# 2) убираем пользователей, которые встречаются в обеих группах
users_multi = df_clean.groupby('user_id')['group'].nunique()
bad_users = users_multi[users_multi > 1].index
df_clean = df_clean[~df_clean['user_id'].isin(bad_users)].copy()

# 3) убираем дубликаты user_id (оставляем первое появление)
df_clean = df_clean.drop_duplicates(subset=['user_id'], keep='first').copy()

df.shape, df_clean.shape

## 4) EDA: баланс групп и конверсия

Смотрим:
- сколько пользователей в каждой группе
- конверсия в control и treatment
- динамика конверсии по дням


In [ ]:
group_counts = df_clean['group'].value_counts()
group_counts

In [ ]:
conv = df_clean.groupby('group')['converted'].mean()
conv

In [ ]:
# Бар-чарт конверсий
plt.bar(conv.index, conv.values)
plt.title("Conversion rate by group")
plt.ylabel("conversion rate")
plt.show()

In [ ]:
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], errors='coerce')
df_clean['date'] = df_clean['timestamp'].dt.date

daily = df_clean.groupby(['date','group'])['converted'].mean().reset_index()
for g in daily['group'].unique():
    d = daily[daily['group']==g]
    plt.plot(d['date'], d['converted'], label=g)

plt.title("Daily conversion rate")
plt.xlabel("date")
plt.ylabel("conversion rate")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 5) Статистический тест: z-test для долей

Гипотезы:
- **H0:** конверсия treatment = конверсия control  
- **H1:** конверсия treatment > конверсия control (рост конверсии)

Уровень значимости: α = 0.05


In [ ]:
alpha = 0.05

control = df_clean[df_clean['group']=='control']
treat   = df_clean[df_clean['group']=='treatment']

successes = np.array([treat['converted'].sum(), control['converted'].sum()])
nobs      = np.array([treat.shape[0], control.shape[0]])

z_stat, p_value = proportions_ztest(count=successes, nobs=nobs, alternative='larger')
z_stat, p_value

In [ ]:
if p_value < alpha:
    print(f"p-value={p_value:.4f} < {alpha}: отвергаем H0. Есть статистически значимый рост конверсии.")
else:
    print(f"p-value={p_value:.4f} >= {alpha}: не отвергаем H0. Статистически значимого роста конверсии нет.")

## 6) Bootstrap CI для разницы конверсий (дополнительно)

Строим доверительный интервал для `treatment - control` на бутстрапе.  
Это помогает “оценить диапазон эффекта”, а не только p-value.


In [ ]:
def bootstrap_diff_ci(df_a, df_b, col='converted', n_iter=5000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    a = df_a[col].to_numpy()
    b = df_b[col].to_numpy()

    diffs = []
    for _ in range(n_iter):
        a_s = rng.choice(a, size=len(a), replace=True).mean()
        b_s = rng.choice(b, size=len(b), replace=True).mean()
        diffs.append(a_s - b_s)

    diffs = np.array(diffs)
    lo = np.quantile(diffs, (1-ci)/2)
    hi = np.quantile(diffs, 1-(1-ci)/2)
    return diffs.mean(), lo, hi

mean_diff, lo, hi = bootstrap_diff_ci(treat, control)
mean_diff, lo, hi

In [ ]:
plt.hist([mean_diff], bins=1)
plt.close()

plt.figure(figsize=(10,4))
# Покажем распределение разницы
diffs = []
rng = np.random.default_rng(42)
a = treat['converted'].to_numpy()
b = control['converted'].to_numpy()
for _ in range(3000):
    diffs.append(rng.choice(a, size=len(a), replace=True).mean() - rng.choice(b, size=len(b), replace=True).mean())
diffs=np.array(diffs)

plt.hist(diffs, bins=40)
plt.axvline(lo, linestyle='--')
plt.axvline(hi, linestyle='--')
plt.axvline(0, linestyle=':')
plt.title("Bootstrap distribution: (treatment - control)")
plt.xlabel("diff in conversion")
plt.show()

print(f"Bootstrap 95% CI: [{lo:.4f}, {hi:.4f}]")

## 7) Итоговый вывод

- После очистки данных группы сбалансированы по размеру.  
- Конверсии близки, и по z-test при α=0.05 **нет** статистически значимого доказательства роста (если p-value ≥ 0.05).  
- Bootstrap CI показывает диапазон возможного эффекта; если 0 лежит внутри CI, то эффект не подтверждён.

**Продуктовое решение:** не выкатывать изменение “всем” только на основании этого теста; лучше:
- уточнить метрику / сегмент, где эффект может быть сильнее  
- проверить power теста (хватило ли выборки)  
- запустить следующий итеративный эксперимент с более сильным изменением
